# SuperNNova transient classifier

PyTorch and WarpTemplate are imported directly from the active kernel. The editable
WarpTemplate installation anchors all samples and generated artifacts in the shared
`data` directory beside the repository. The recurrent model is local to WarpTemplate; an external SuperNNova
installation is not required for training.


In [ ]:
# Configure paths and import PyTorch plus the local recurrent-classifier backend.
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Image

import warptemplate
from warptemplate import classification as workflow
from warptemplate import supernnova_backend as supernova

# Anchor shared data beside the editable checkout, independent of the kernel cwd.
DATA_ROOT = Path(warptemplate.__file__).resolve().parents[2] / "data"
sns.set_theme(context="notebook", style="ticks")


## Configuration

Use a readable `RUN_NAME` and explicit actions. `run` creates a new artifact, `load`
reads its fixed path, `resume` exactly continues an incomplete SuperNNova training
checkpoint, and `skip` leaves a stage untouched. Existing official test artifacts are
never overwritten; use another `RUN_NAME` for a new experiment.


In [ ]:
# Define user settings first, then derive every fixed artifact path.
# ==============================================================================
# USER SETTINGS — EDIT VALUES HERE, THEN SELECT "RUN ALL"
# ==============================================================================
SAMPLE_ID = "warp_sample_combined_schema6_660649e34711"
RUN_NAME = "supernnova_baseline"
SPLIT_STRATEGY = "basis_sn"
EXECUTION_MODE = "smoke"              # smoke, full_cpu, full_gpu_repeats
SEED = 20260723
THREADS = 14
BIDIRECTIONAL = True
POOLING = "mean"                       # mean, standard, attention
USE_ENGINEERED_FEATURES = False
PARTIAL_CUTOFFS = [-7, -2, -1, 0, 1, 2, 30]

SPLIT_ACTION = "run"                  # run, load
DATABASE_ACTION = "run"               # run, load
TRAIN_ACTION = "run"                  # run, resume, load, skip
SMOKE_EVALUATION_ACTION = "run"       # run, load, skip
TEST_ACTION = "skip"                   # run, load, skip

MODE_CONFIGS = {
    "smoke": {
        "objects_per_class": 128,
        "epochs": 20,
        "device": "cpu",
        "seeds": [SEED],
    },
    "full_cpu": {
        "objects_per_class": None,
        "epochs": 90,
        "device": "cpu",
        "seeds": [SEED],
    },
    "full_gpu_repeats": {
        "objects_per_class": None,
        "epochs": 90,
        "device": "cuda",
        "seeds": list(range(20260721, 20260726)),
    },
}

# ==============================================================================
# DERIVED CONFIGURATION — DO NOT EDIT PATHS BELOW
# ==============================================================================
mode_config = MODE_CONFIGS[EXECUTION_MODE]
RUN_SEEDS = mode_config["seeds"]
DEVICE = mode_config["device"]
MAX_EPOCHS = mode_config["epochs"]
SMOKE_OBJECTS_PER_CLASS = mode_config["objects_per_class"] or 128
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("The selected execution mode requires CUDA")

SAMPLE_DIR = DATA_ROOT / "training_samples" / SAMPLE_ID
SPLIT_ROOT = DATA_ROOT / "classification_splits" / SAMPLE_ID
RUN_ROOT = (
    DATA_ROOT / "classifier_runs" / SAMPLE_ID / SPLIT_STRATEGY
    / "supernnova" / RUN_NAME
)
DATABASE_PATH = RUN_ROOT / "database" / EXECUTION_MODE / "database.h5"
REDSHIFT_MODES = list(supernova.REDSHIFT_MODES)
workflow.set_random_seed(SEED)
print(
    {
        "mode": EXECUTION_MODE,
        "device": DEVICE,
        "seeds": RUN_SEEDS,
        "epochs": MAX_EPOCHS,
        "input": str(SAMPLE_DIR),
        "output": str(RUN_ROOT),
    }
)


## Frozen groups and sequence database

Both variants share one raw HDF5 database. It stores the 28 baseline photometry channels, opt-in engineered S/N, detection, limiting-depth, cumulative-time, and band-coverage channels, plus exact simulated redshift. Redshift is fused once after sequence pooling rather than repeated through the LSTM. Fluxes are converted to zeropoint 27.5, measurements within 0.33 days are grouped by inverse variance, and mask-aware normalization is learned from observed values in the active training role only.

In [ ]:
# Create or directly load the selected split and sequence database.
truth = workflow.load_sample_truth(SAMPLE_DIR)
truth["final_label"] = workflow.merge_fitclasses(truth["fitclass"]).to_numpy()
split_manifest, excluded = workflow.prepare_persistent_split(
    SAMPLE_DIR,
    SPLIT_ROOT,
    strategy=SPLIT_STRATEGY,
    seed=SEED,
    action=SPLIT_ACTION,
)
workflow.validate_split_manifest(split_manifest)
role_manifest = supernova.build_execution_role_manifest(
    split_manifest,
    EXECUTION_MODE,
    objects_per_class=SMOKE_OBJECTS_PER_CLASS,
    seed=SEED,
)
display(pd.crosstab(role_manifest["final_label"], role_manifest["role"], margins=True))

if DATABASE_ACTION == "run":
    database_summary = supernova.prepare_supernnova_database(
        SAMPLE_DIR,
        truth,
        role_manifest,
        DATABASE_PATH,
    )
elif DATABASE_ACTION == "load":
    database_summary = supernova.load_supernnova_database_summary(DATABASE_PATH)
else:
    raise ValueError("DATABASE_ACTION must be 'run' or 'load'")
display(pd.Series(database_summary).drop("feature_order"))
print(f"Sequence database: {DATABASE_PATH}")


## Train or reload both variants

The optimizer sees inverse-frequency weighted cross-entropy. Complete validation sequences select the best checkpoint by class-balanced log loss. Full runs halve the learning rate after five unimproved epochs and stop after twelve; the smoke run always has a two-epoch ceiling.

In [ ]:
# Run, exactly resume, directly load, or skip every requested recurrent model.
runs = {}
for run_seed in RUN_SEEDS:
    for redshift_mode in REDSHIFT_MODES:
        training_config = supernova.SuperNNovaTrainingConfig(
            redshift_mode=redshift_mode,
            seed=run_seed,
            epochs=MAX_EPOCHS,
            bidirectional=BIDIRECTIONAL,
            rnn_output_option=POOLING,
            engineered_features=USE_ENGINEERED_FEATURES,
            device=DEVICE,
            threads=THREADS,
        )
        variant_run_id = f"{RUN_NAME}_{redshift_mode}_seed{run_seed}"
        experiment_config = workflow.ExperimentConfig(
            training_sample=SAMPLE_ID,
            evaluation_sample=SAMPLE_ID,
            backend="supernnova",
            redshift_mode=redshift_mode,
            split_strategy=SPLIT_STRATEGY,
            seed=run_seed,
            model_config={
                **training_config.normalized(),
                "execution_mode": EXECUTION_MODE,
                "database_identity": database_summary["database_identity"],
            },
            run_id=variant_run_id,
        )
        variant_dir = (
            RUN_ROOT / redshift_mode / f"seed_{run_seed}" / EXECUTION_MODE
        )
        model_dir = variant_dir / "models"
        experiment_path = variant_dir / "experiment.json"
        checkpoint_path = model_dir / "best.pt"
        history = None

        if TRAIN_ACTION == "run":
            workflow.ensure_run_metadata(
                experiment_config, split_manifest, experiment_path
            )
            history = supernova.train_supernnova(
                DATABASE_PATH,
                model_dir,
                training_config,
                action="run",
                show_progress=True,
            )
        elif TRAIN_ACTION == "resume":
            workflow.load_run_metadata(experiment_path, experiment_config)
            history = supernova.train_supernnova(
                DATABASE_PATH,
                model_dir,
                training_config,
                action="resume",
                show_progress=True,
            )
        elif TRAIN_ACTION == "load":
            workflow.load_run_metadata(experiment_path, experiment_config)
            history = supernova.load_supernnova_training_history(model_dir)
            _, loaded_config = supernova.load_supernnova_checkpoint(
                checkpoint_path, device=DEVICE
            )
            if loaded_config.normalized() != training_config.normalized():
                raise ValueError("Checkpoint training configuration does not match this run")
        elif TRAIN_ACTION != "skip":
            raise ValueError("TRAIN_ACTION must be 'run', 'resume', 'load', or 'skip'")

        runs[(run_seed, redshift_mode)] = {
            "config": experiment_config,
            "training_config": training_config,
            "variant_dir": variant_dir,
            "history": history,
            "checkpoint": checkpoint_path,
            "experiment_path": experiment_path,
        }
        if history is not None:
            print(
                redshift_mode,
                run_seed,
                f"{history['elapsed_seconds'] / 60:.2f} min",
                variant_dir,
            )
if TRAIN_ACTION == "skip":
    print("Training stage skipped.")


## Smoke validation and preliminary confusion matrices

Smoke validation uses fold 3 and the disposable smoke test uses fold 2. Neither is the official fold-0 test. With 64 objects per class, row-normalized confusion matrices make the preliminary class recall directly comparable across variants.

In [ ]:
# Run, directly load, or skip smoke validation and its disposable smoke test.
from sklearn.metrics import confusion_matrix

smoke_predictions = {}
smoke_metrics = {}
smoke_figure_path = RUN_ROOT / "smoke" / "confusion_matrices.png"
if SMOKE_EVALUATION_ACTION == "run":
    if EXECUTION_MODE != "smoke":
        raise ValueError("Smoke evaluation requires EXECUTION_MODE='smoke'")
    smoke_manifest = role_manifest.copy()
    smoke_manifest["split"] = smoke_manifest["role"].map(
        {
            "train": "smoke_train",
            "validation": "smoke_validation",
            "test": "smoke_test",
        }
    )
    for redshift_mode in REDSHIFT_MODES:
        run = runs[(SEED, redshift_mode)]
        for role, partition in (
            ("validation", "smoke_validation"),
            ("test", "smoke_test"),
        ):
            prediction_path = run["variant_dir"] / "predictions" / f"{partition}.parquet"
            metric_path = run["variant_dir"] / "metrics" / f"{partition}.json"
            result = supernova.predict_supernnova(
                run["checkpoint"], DATABASE_PATH, role=role, device=DEVICE
            )
            predictions = workflow.standardize_predictions(
                result.classifications,
                smoke_manifest,
                run["config"],
                partition=partition,
            )
            metrics = workflow.compute_classification_metrics(predictions)
            workflow.write_table_once(predictions, prediction_path)
            workflow.write_json_once(metrics, metric_path)
            smoke_predictions[(redshift_mode, partition)] = predictions
            smoke_metrics[(redshift_mode, partition)] = metrics

    figure, axes = plt.subplots(2, 2, figsize=(15, 12))
    for row, redshift_mode in enumerate(REDSHIFT_MODES):
        for column, partition in enumerate(("smoke_validation", "smoke_test")):
            predictions = smoke_predictions[(redshift_mode, partition)]
            matrix = confusion_matrix(
                predictions["true_class"],
                predictions["predicted_class"],
                labels=workflow.FINAL_CLASSES,
                normalize="true",
            )
            sns.heatmap(
                matrix,
                annot=True,
                fmt=".2f",
                vmin=0,
                vmax=1,
                cmap="Blues",
                xticklabels=workflow.FINAL_CLASSES,
                yticklabels=workflow.FINAL_CLASSES,
                ax=axes[row, column],
            )
            axes[row, column].set(
                title=f"{redshift_mode}: {partition.replace('_', ' ')}",
                xlabel="Predicted class",
                ylabel="True class",
            )
    figure.tight_layout()
    workflow.save_figure_once(
        figure, smoke_figure_path, dpi=180, bbox_inches="tight"
    )
    plt.close(figure)

elif SMOKE_EVALUATION_ACTION == "load":
    if EXECUTION_MODE != "smoke":
        raise ValueError("Smoke evaluation requires EXECUTION_MODE='smoke'")
    for redshift_mode in REDSHIFT_MODES:
        run = runs[(SEED, redshift_mode)]
        workflow.load_run_metadata(run["experiment_path"], run["config"])
        for partition in ("smoke_validation", "smoke_test"):
            prediction_path = run["variant_dir"] / "predictions" / f"{partition}.parquet"
            metric_path = run["variant_dir"] / "metrics" / f"{partition}.json"
            smoke_predictions[(redshift_mode, partition)] = pd.read_parquet(
                prediction_path
            )
            smoke_metrics[(redshift_mode, partition)] = json.loads(
                metric_path.read_text()
            )
elif SMOKE_EVALUATION_ACTION != "skip":
    raise ValueError(
        "SMOKE_EVALUATION_ACTION must be 'run', 'load', or 'skip'"
    )

if SMOKE_EVALUATION_ACTION != "skip":
    display(Image(filename=str(smoke_figure_path)))
    metric_rows = []
    for (redshift_mode, partition), metrics in smoke_metrics.items():
        metric_rows.append(
            {
                "variant": redshift_mode,
                "partition": partition,
                **{
                    key: metrics[key]
                    for key in (
                        "class_balanced_log_loss", "balanced_accuracy", "macro_f1",
                        "top_1_accuracy", "top_2_accuracy", "multiclass_brier",
                    )
                },
            }
        )
    display(pd.DataFrame(metric_rows).set_index(["variant", "partition"]))
else:
    print("Smoke evaluation skipped; official fold 0 remains untouched.")


## Frozen full evaluation and partial light curves

This section remains inactive in the default smoke run. In a full mode, enabling `EVALUATE_TEST` writes immutable fold-0 probabilities, the shared metric suite and group bootstrap, observing-condition breakdowns, calibration diagnostics, and peak-relative partial classifications. An object with no measurement before a partial cutoff is counted as ineligible rather than padded with invented data.

In [ ]:
# Run once, directly load, or skip full and partial official-test predictions.
full_predictions = {}
partial_summaries = []
if TEST_ACTION == "run":
    if EXECUTION_MODE == "smoke":
        raise ValueError("Official test evaluation requires a full execution mode")
    test_observations = workflow.select_partition_rows(
        SAMPLE_DIR, split_manifest, "test"
    )
    for (run_seed, redshift_mode), run in runs.items():
        prediction_path = run["variant_dir"] / "predictions" / "test.parquet"
        metric_path = run["variant_dir"] / "metrics" / "test.json"
        breakdown_path = run["variant_dir"] / "metrics" / "test_breakdowns.parquet"
        partial_summary_path = run["variant_dir"] / "metrics" / "partial_summary.parquet"
        result = supernova.predict_supernnova(
            run["checkpoint"], DATABASE_PATH, role="test", device=DEVICE
        )
        predictions = workflow.standardize_predictions(
            result.classifications, split_manifest, run["config"], partition="test"
        )
        metrics = workflow.compute_classification_metrics(predictions)
        metrics["group_bootstrap_95"] = workflow.group_bootstrap_confidence_intervals(
            predictions, split_manifest, repeats=1000, seed=run_seed
        )
        breakdowns = workflow.metric_breakdowns(
            predictions, truth, test_observations
        )
        workflow.write_table_once(predictions, prediction_path)
        workflow.write_json_once(metrics, metric_path)
        workflow.write_table_once(breakdowns, breakdown_path)
        full_predictions[(run_seed, redshift_mode)] = predictions

        variant_partial_rows = []
        for cutoff in PARTIAL_CUTOFFS:
            partial_result = supernova.predict_supernnova(
                run["checkpoint"],
                DATABASE_PATH,
                role="test",
                cutoff_days=cutoff,
                device=DEVICE,
            )
            partial_metrics = {}
            if partial_result.eligible_objects:
                partial = workflow.standardize_predictions(
                    partial_result.classifications,
                    split_manifest,
                    run["config"],
                    partition="test",
                )
                partial_metrics = workflow.compute_classification_metrics(partial)
                partial["cutoff_days"] = cutoff
                workflow.write_table_once(
                    partial,
                    run["variant_dir"]
                    / "predictions"
                    / f"test_cutoff_{cutoff:+d}.parquet",
                )
            variant_partial_rows.append(
                {
                    "seed": run_seed,
                    "variant": redshift_mode,
                    "cutoff_days": cutoff,
                    "eligible": partial_result.eligible_objects,
                    "ineligible": partial_result.ineligible_objects,
                    **{
                        name: partial_metrics.get(name, np.nan)
                        for name in (
                            "class_balanced_log_loss", "balanced_accuracy",
                            "macro_f1", "top_2_accuracy",
                        )
                    },
                }
            )
        variant_partial = pd.DataFrame(variant_partial_rows)
        workflow.write_table_once(variant_partial, partial_summary_path)
        partial_summaries.extend(variant_partial_rows)

elif TEST_ACTION == "load":
    if EXECUTION_MODE == "smoke":
        raise ValueError("Official test evaluation requires a full execution mode")
    for key, run in runs.items():
        workflow.load_run_metadata(run["experiment_path"], run["config"])
        full_predictions[key] = pd.read_parquet(
            run["variant_dir"] / "predictions" / "test.parquet"
        )
        partial_summaries.extend(
            pd.read_parquet(
                run["variant_dir"] / "metrics" / "partial_summary.parquet"
            ).to_dict("records")
        )
elif TEST_ACTION != "skip":
    raise ValueError("TEST_ACTION must be 'run', 'load', or 'skip'")
else:
    print("Official test evaluation skipped.")

if partial_summaries:
    display(pd.DataFrame(partial_summaries))


In [ ]:
# Run or directly load paired comparisons and full-test diagnostic figures.
if full_predictions:
    from sklearn.metrics import confusion_matrix

    for run_seed in RUN_SEEDS:
        comparison_path = RUN_ROOT / "comparisons" / f"paired_seed{run_seed}.json"
        if TEST_ACTION == "run":
            paired = supernova.paired_group_bootstrap_differences(
                full_predictions[(run_seed, "photometry_only")],
                full_predictions[(run_seed, "photometry_plus_truth_z")],
                split_manifest,
                repeats=1000,
                seed=run_seed,
            )
            workflow.write_json_once(paired, comparison_path)
        else:
            paired = json.loads(comparison_path.read_text())
        display(pd.DataFrame(paired["intervals"]).T)

    for key, predictions in full_predictions.items():
        run = runs[key]
        figure_path = run["variant_dir"] / "figures" / "test_diagnostics.png"
        if TEST_ACTION == "run":
            metrics = workflow.compute_classification_metrics(predictions)
            probabilities = predictions[
                [f"prob_{label}" for label in workflow.FINAL_CLASSES]
            ].to_numpy()
            entropy = -(
                probabilities * np.log(np.clip(probabilities, 1e-15, 1))
            ).sum(axis=1)
            raw = confusion_matrix(
                predictions["true_class"],
                predictions["predicted_class"],
                labels=workflow.FINAL_CLASSES,
            )
            normalized = confusion_matrix(
                predictions["true_class"],
                predictions["predicted_class"],
                labels=workflow.FINAL_CLASSES,
                normalize="true",
            )
            figure, axes = plt.subplots(2, 2, figsize=(14, 11))
            sns.heatmap(
                raw,
                annot=True,
                fmt="d",
                xticklabels=workflow.FINAL_CLASSES,
                yticklabels=workflow.FINAL_CLASSES,
                ax=axes[0, 0],
            )
            sns.heatmap(
                normalized,
                annot=True,
                fmt=".2f",
                vmin=0,
                vmax=1,
                xticklabels=workflow.FINAL_CLASSES,
                yticklabels=workflow.FINAL_CLASSES,
                ax=axes[0, 1],
            )
            calibration = pd.DataFrame(metrics["calibration"])
            axes[1, 0].plot([0, 1], [0, 1], "k--", label="Ideal")
            axes[1, 0].plot(
                calibration["confidence"], calibration["accuracy"], "o-", label="Model"
            )
            axes[1, 0].legend()
            sns.histplot(entropy, bins=30, ax=axes[1, 1])
            figure.suptitle(f"SuperNNova {key[1]} seed {key[0]}")
            figure.tight_layout()
            workflow.save_figure_once(
                figure, figure_path, dpi=180, bbox_inches="tight"
            )
            plt.close(figure)
        display(Image(filename=str(figure_path)))
else:
    print("Comparison stage waits for a run or load of official test predictions.")


## Interpretation

The smoke result verifies preprocessing, optimization, checkpoint reload, and standardized probabilities; two epochs are not expected to converge. Compare full scientific results only when the evaluation sample and split strategy match, and treat `photometry_plus_truth_z` as an optimistic exact-redshift baseline.